# Long version of UHF code

In [ ]:
import numpy as np
from math import pi, exp, sqrt, erf

# ===========================================================================
# Gaussian integral routines
# ===========================================================================

def norm(alpha):
    return (2.0 * alpha / pi) ** 0.75

def F0(t):
    """Boys function of order 0"""
    if t < 1e-7:
        return 1.0
    return 0.5 * sqrt(pi / t) * erf(sqrt(t))

def overlap_int(alpha, A, beta, B):
    AB2 = sum((a - b)**2 for a, b in zip(A, B))
    p   = alpha + beta
    return norm(alpha) * norm(beta) * (pi / p)**1.5 * exp(-alpha * beta / p * AB2)

def kinetic_int(alpha, A, beta, B):
    AB2 = sum((a - b)**2 for a, b in zip(A, B))
    p   = alpha + beta
    pre = norm(alpha) * norm(beta) * (pi / p)**1.5 * exp(-alpha * beta / p * AB2)
    return alpha * beta / p * (3.0 - 2.0 * alpha * beta / p * AB2) * pre

def nuclear_int(alpha, A, beta, B, Z, C):
    AB2 = sum((a - b)**2 for a, b in zip(A, B))
    p   = alpha + beta
    P   = [(alpha * a + beta * b) / p for a, b in zip(A, B)]
    PC2 = sum((pi_ - c)**2 for pi_, c in zip(P, C))
    return (norm(alpha) * norm(beta) * (-2.0 * pi * Z / p)
            * F0(p * PC2) * exp(-alpha * beta / p * AB2))

def eri(alpha, A, beta, B, gamma, C, delta, D):
    AB2  = sum((a - b)**2 for a, b in zip(A, B))
    CD2  = sum((a - b)**2 for a, b in zip(C, D))
    p    = alpha + beta
    q    = gamma + delta
    P    = [(alpha * a + beta * b) / p for a, b in zip(A, B)]
    Q    = [(gamma * c + delta * d) / q for c, d in zip(C, D)]
    PQ2  = sum((a - b)**2 for a, b in zip(P, Q))
    zeta = p * q / (p + q)
    pre  = (norm(alpha) * norm(beta) * norm(gamma) * norm(delta)
            * 2.0 * pi**2.5 / (p * q * sqrt(p + q))
            * exp(-alpha * beta / p * AB2 - gamma * delta / q * CD2))
    return pre * F0(zeta * PQ2)

def eint(a, b, c, d):
    ab = int(a*(a+1)/2 + b) if a > b else int(b*(b+1)/2 + a)
    cd = int(c*(c+1)/2 + d) if c > d else int(d*(d+1)/2 + c)
    return int(ab*(ab+1)/2 + cd) if ab > cd else int(cd*(cd+1)/2 + ab)

def tei(a, b, c, d):
    return twoe.get(eint(a, b, c, d), 0.0)

# ===========================================================================
# UHF routines
# ===========================================================================

def make_uhf_fock(Hcore, P_alpha, P_beta, dim):
    P_total = P_alpha + P_beta
    F_alpha = np.zeros((dim, dim))
    F_beta  = np.zeros((dim, dim))
    for mu in range(dim):
        for nu in range(dim):
            F_alpha[mu, nu] = Hcore[mu, nu]
            F_beta[mu, nu]  = Hcore[mu, nu]
            for lam in range(dim):
                for sig in range(dim):
                    coulomb = tei(mu+1, nu+1, sig+1, lam+1)
                    exch    = tei(mu+1, lam+1, sig+1, nu+1)
                    F_alpha[mu, nu] += P_total[lam, sig] * coulomb - P_alpha[lam, sig] * exch
                    F_beta[mu, nu]  += P_total[lam, sig] * coulomb - P_beta[lam, sig]  * exch
    return F_alpha, F_beta

def transform_and_diag(F, X):
    Fprime      = np.dot(X.T, np.dot(F, X))
    eps, Cprime = np.linalg.eigh(Fprime)
    C           = np.dot(X, Cprime)
    return eps, C

def make_density_uhf(C, N_occ, dim):
    D = np.zeros((dim, dim))
    for mu in range(dim):
        for nu in range(dim):
            for m in range(N_occ):
                D[mu, nu] += C[mu, m] * C[nu, m]
    return D

def uhf_energy(P_alpha, P_beta, Hcore, F_alpha, F_beta, dim):
    E = 0.0
    for mu in range(dim):
        for nu in range(dim):
            E += 0.5 * P_alpha[mu, nu] * (Hcore[mu, nu] + F_alpha[mu, nu])
            E += 0.5 * P_beta[mu, nu]  * (Hcore[mu, nu] + F_beta[mu, nu])
    return E

def rmsd(D_new, D_old, dim):
    delta = 0.0
    for mu in range(dim):
        for nu in range(dim):
            delta += (D_new[mu, nu] - D_old[mu, nu])**2
    return delta**0.5

def spin_contamination(C_alpha, C_beta, S, Na, Nb):
    S_exact  = (Na - Nb) / 2.0
    S2_exact = S_exact * (S_exact + 1.0)
    dim      = S.shape[0]
    O = np.zeros((Na, Nb))
    for i in range(Na):
        for j in range(Nb):
            for mu in range(dim):
                for nu in range(dim):
                    O[i, j] += C_alpha[mu, i] * S[mu, nu] * C_beta[nu, j]
    S2_uhf      = S2_exact + Nb - np.sum(O**2)
    spin_contam = S2_uhf - S2_exact
    return S2_uhf, S2_exact, spin_contam

# ===========================================================================
# System specification: HeH radical, STO-1G
# ===========================================================================

Na   = 2       # alpha electrons
Nb   = 1       # beta electrons
ENUC = 1.3230  # nuclear repulsion (hartrees)
dim  = 2       # number of basis functions
TOL  = 1e-5    # SCF convergence threshold

alphas  = [0.4166, 0.7739]
centers = [[0.0, 0.0, 0.0], [0.0, 0.0, 1.5117]]
nuclei  = [(1, centers[0]), (2, centers[1])]   # (Z, position)

# ===========================================================================
# Build one-electron matrices
# ===========================================================================

S     = np.zeros((dim, dim))
T     = np.zeros((dim, dim))
V     = np.zeros((dim, dim))

for mu in range(dim):
    for nu in range(dim):
        S[mu, nu] = overlap_int(alphas[mu], centers[mu], alphas[nu], centers[nu])
        T[mu, nu] = kinetic_int(alphas[mu], centers[mu], alphas[nu], centers[nu])
        for (Z, Rc) in nuclei:
            V[mu, nu] += nuclear_int(alphas[mu], centers[mu],
                                     alphas[nu], centers[nu], Z, Rc)

S     = 0.5 * (S + S.T)
T     = 0.5 * (T + T.T)
V     = 0.5 * (V + V.T)
Hcore = T + V

# ===========================================================================
# Build two-electron integral table
# ===========================================================================

twoe = {}
for mu in range(dim):
    for nu in range(dim):
        for lam in range(dim):
            for sig in range(dim):
                key = eint(mu+1, nu+1, lam+1, sig+1)
                if key not in twoe:
                    twoe[key] = eri(alphas[mu], centers[mu],
                                    alphas[nu], centers[nu],
                                    alphas[lam], centers[lam],
                                    alphas[sig], centers[sig])

# ===========================================================================
# Form transformation matrix X = S^{-1/2}
# ===========================================================================

SVAL, SVEC   = np.linalg.eigh(S)
SVAL_minhalf = np.diag(SVAL**(-0.5))
X            = np.dot(SVEC, np.dot(SVAL_minhalf, SVEC.T))

# ===========================================================================
# Initialise density matrices and run UHF SCF
# ===========================================================================

P_alpha = np.zeros((dim, dim))
P_beta  = np.zeros((dim, dim))
E_old   = 0.0
count   = 0

while True:
    count += 1
    F_alpha, F_beta = make_uhf_fock(Hcore, P_alpha, P_beta, dim)

    eps_alpha, C_alpha = transform_and_diag(F_alpha, X)
    eps_beta,  C_beta  = transform_and_diag(F_beta,  X)

    P_alpha_new = make_density_uhf(C_alpha, Na, dim)
    P_beta_new  = make_density_uhf(C_beta,  Nb, dim)

    E_el = uhf_energy(P_alpha_new, P_beta_new, Hcore, F_alpha, F_beta, dim)

    dE        = abs(E_el - E_old)
    rms_alpha = rmsd(P_alpha_new, P_alpha, dim)
    rms_beta  = rmsd(P_beta_new,  P_beta,  dim)

    print("E = {:.8f}  dE = {:.2e}  rmsP_a = {:.2e}  rmsP_b = {:.2e}  N(SCF) = {}".format(
          E_el + ENUC, dE, rms_alpha, rms_beta, count))

    if dE < TOL and rms_alpha < TOL and rms_beta < TOL and count > 1:
        break

    P_alpha = P_alpha_new
    P_beta  = P_beta_new
    E_old   = E_el

# ===========================================================================
# Final results
# ===========================================================================

print("\nSCF converged!")
print("UHF electronic energy = {:.8f} hartrees".format(E_el))
print("Nuclear repulsion      = {:.8f} hartrees".format(ENUC))
print("UHF total energy       = {:.8f} hartrees".format(E_el + ENUC))

print("\nAlpha orbital energies: {}".format(eps_alpha))
print("Beta  orbital energies: {}".format(eps_beta))

S2_uhf, S2_exact, contam = spin_contamination(C_alpha, C_beta, S, Na, Nb)
print("\n<S^2> expected     = {:.4f}".format(S2_exact))
print("<S^2> UHF          = {:.4f}".format(S2_uhf))
print("Spin contamination = {:.4f}".format(contam))


# Short version of UHF code


In [ ]:
import numpy as np
from math import pi, exp, sqrt, erf

def norm(a): 
    return (2.0 * a / pi) ** 0.75

def F0(t): 
    return 1.0 if t < 1e-7 else 0.5 * sqrt(pi / t) * erf(sqrt(t))
    
def dist2(A, B): 
    return sum((a - b)**2 for a, b in zip(A, B))

def overlap_int(a, A, b, B):
    return norm(a) * norm(b) * (pi / (a + b))**1.5 * exp(-a * b / (a + b) * dist2(A, B))

def kinetic_int(a, A, b, B):
    p = a + b
    return (a * b / p * (3.0 - 2.0 * a * b / p * dist2(A, B))) * overlap_int(a, A, b, B)

def nuclear_int(a, A, b, B, Z, C):
    p, P = a + b, [(a * x + b * y) / (a + b) for x, y in zip(A, B)]
    return norm(a)*norm(b) * (-2.0*pi*Z/p) * F0(p*dist2(P, C)) * exp(-a*b/p*dist2(A, B))

def eri(a, A, b, B, c, C, d, D):
    p, q = a + b, c + d
    P, Q = [(a*x + b*y)/p for x, y in zip(A, B)], [(c*x + d*y)/q for x, y in zip(C, D)]
    pre = norm(a)*norm(b)*norm(c)*norm(d) * 2.0*pi**2.5 / (p*q*sqrt(p+q)) * exp(-a*b/p*dist2(A,B) - c*d/q*dist2(C,D))
    return pre * F0((p * q / (p + q)) * dist2(P, Q))

def eint(a, b, c, d):
    ab = a*(a+1)//2 + b if a > b else b*(b+1)//2 + a
    cd = c*(c+1)//2 + d if c > d else d*(d+1)//2 + c
    return ab*(ab+1)//2 + cd if ab > cd else cd*(cd+1)//2 + ab

# System specification
Na, Nb, ENUC, dim, TOL = 2, 1, 1.3230, 2, 1e-5
alphas, centers = [0.4166, 0.7739], [[0.0, 0.0, 0.0], [0.0, 0.0, 1.5117]]
nuclei = [(1, centers[0]), (2, centers[1])]

# Matrix Initialization
S, T, V = np.zeros((dim, dim)), np.zeros((dim, dim)), np.zeros((dim, dim))
for i, j in np.ndindex(dim, dim):
    S[i, j] = overlap_int(alphas[i], centers[i], alphas[j], centers[j])
    T[i, j] = kinetic_int(alphas[i], centers[i], alphas[j], centers[j])
    V[i, j] = sum(nuclear_int(alphas[i], centers[i], alphas[j], centers[j], Z, Rc) for Z, Rc in nuclei)

Hcore, S = 0.5*(T+T.T) + 0.5*(V+V.T), 0.5*(S+S.T)
twoe = {eint(i+1, j+1, k+1, l+1): eri(alphas[i], centers[i], alphas[j], centers[j], 
        alphas[k], centers[k], alphas[l], centers[l]) for i, j, k, l in np.ndindex(dim, dim, dim, dim)}

# SCF Setup
val, vec = np.linalg.eigh(S)
X = vec @ np.diag(val**(-0.5)) @ vec.T
Pa, Pb, E_old, count = np.zeros((dim, dim)), np.zeros((dim, dim)), 0.0, 0

while True:
    count += 1
    Fa, Fb = Hcore.copy(), Hcore.copy()
    for i, j, k, l in np.ndindex(dim, dim, dim, dim):
        coul, exch = twoe[eint(i+1,j+1,l+1,k+1)], twoe[eint(i+1,k+1,l+1,j+1)]
        Fa[i, j] += (Pa+Pb)[k, l] * coul - Pa[k, l] * exch
        Fb[i, j] += (Pa+Pb)[k, l] * coul - Pb[k, l] * exch

    ea, Ca = np.linalg.eigh(X.T @ Fa @ X); Ca = X @ Ca
    eb, Cb = np.linalg.eigh(X.T @ Fb @ X); Cb = X @ Cb
    Pa_new, Pb_new = Ca[:, :Na] @ Ca[:, :Na].T, Cb[:, :Nb] @ Cb[:, :Nb].T
    E_el = 0.5 * np.sum(Pa_new * (Hcore + Fa)) + 0.5 * np.sum(Pb_new * (Hcore + Fb))
    
    dE, rms_a, rms_b = abs(E_el - E_old), np.linalg.norm(Pa_new - Pa), np.linalg.norm(Pb_new - Pb)
    print(f"E = {E_el+ENUC:.8f}  dE = {dE:.2e}  rmsP_a = {rms_a:.2e}  rmsP_b = {rms_b:.2e}  N = {count}")
    if dE < TOL and rms_a < TOL and rms_b < TOL and count > 1: break
    Pa, Pb, E_old = Pa_new, Pb_new, E_el

# Final Results
S2_ex = (Na - Nb) / 2.0 * ((Na - Nb) / 2.0 + 1.0)
S2_uhf = S2_ex + Nb - np.sum((Ca[:, :Na].T @ S @ Cb[:, :Nb])**2)

print(f"\nSCF converged!\nTotal Energy = {E_el + ENUC:.8f} hartrees")
print(f"Alpha/Beta orbital energies: {ea} / {eb}\nSpin Contamination = {S2_uhf - S2_ex:.4f}")